# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

---
## 0. Подготовка

In [2]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [3]:
# ============================================================
# Настройка LLM-клиентов
# ============================================================

client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o"

---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [4]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+ 
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [5]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

In [6]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

✅ Подключение к GitHub MCP установлено!

🔧 Инструменты (26):
-------------------------------------------------------
  • create_or_update_file
    Create or update a single file in a GitHub repository...

  • search_repositories
    Search for GitHub repositories...

  • create_repository
    Create a new GitHub repository in your account...

  • get_file_contents
    Get the contents of a file or directory from a GitHub repository...

  • push_files
    Push multiple files to a GitHub repository in a single commit...

  • create_issue
    Create a new issue in a GitHub repository...

  • create_pull_request
    Create a new pull request in a GitHub repository...

  • fork_repository
    Fork a GitHub repository to your account or specified organization...

  • create_branch
    Create a new branch in a GitHub repository...

  • list_commits
    Get list of commits of a branch in a GitHub repository...

  • list_issues
    List issues in a GitHub repository with filtering options...

 

---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [7]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


# --- Загрузка ---
skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills_solved"
SKILLS = load_skills(skills_dir)

print(f"\U0001f4c2 Директория skills: {skills_dir}")
print(f"\U0001f4cb Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"  \u2022 {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...")
    print()

📂 Директория skills: C:\Coding\HSE-Agent-Systems_2026\lectures\lecture_3\hw_3_skills_solved
📋 Загружено skills: 3

  • analysis_report: Analysis Report
    Триггер: Use this skill when the user asks for a consolidated report across multiple save...

  • prompt_analysis: Prompt Analysis
    Триггер: Use this skill when the user asks to analyze one specific prompt file from GitHu...

  • prompt_generator: Prompt Generator
    Триггер: Use this skill when the user asks to generate a new system prompt based on the c...



In [8]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills registry for HW 3

This file is Level 1 metadata only.
Use it in the system prompt so the agent sees a compact overview instead of full procedures.
Load a full skill with `read_skill(name)` only when the current task clearly matches that skill.

## Available skills

### prompt_analysis
Use when the task is to inspect one specific prompt file from GitHub, identify its structure and prompting techniques, evaluate quality, and save a per-file analysis with `save_analysis(name, content)`.

### analysis_report
Use when the task is to read all saved analyses, compare recurring patterns across prompts, extract reusable design principles, produce one consolidated report, and save it with `save_report(name, content)`.

### prompt_generator
Use when the task is to read the consolidated report, synthesize the best patterns into a new system prompt, explain major design choices briefly, and save the result with `sav

In [9]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


# --- Тест ---
print("\U0001f527 READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

🔧 READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Prompt Analysis ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Prompt Analysis

## Когда активировать
Use this skill when the user asks to analyze one specific prompt file from GitHub and save a structured per-file analysis.
Typical signals: "analyze this prompt", "inspect prompt.md", "review system prompt", "save analysis for this file".
Use exactly one file per age...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [10]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [11]:
# ============================================================
# Инструменты агента
# ============================================================

import json
import re

# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Директория анализов: {ANALYSES_DIR}")
print(f"📂 Директория отчётов:  {REPORTS_DIR}")


def _safe_name(name: str) -> str:
    """Санитизировать имя для локального файла."""
    cleaned = re.sub(r"[^\w\-./]+", "_", str(name).strip(), flags=re.UNICODE)
    cleaned = cleaned.replace("/", "__").replace("\\", "__")
    cleaned = cleaned.replace(".md", "")
    cleaned = re.sub(r"_+", "_", cleaned).strip("._")
    return cleaned or "untitled"


def _clip(text: str, limit: int = 240) -> str:
    text = (text or "").strip()
    if len(text) <= limit:
        return text
    return text[:limit].rstrip() + "..."


def _extract_last_json_block(text: str) -> dict:
    """Достать последний fenced json block из markdown."""
    if not isinstance(text, str):
        return {}

    blocks = re.findall(r"```json\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    for block in reversed(blocks):
        try:
            data = json.loads(block)
            if isinstance(data, dict):
                return data
        except Exception:
            continue
    return {}


def _extract_section(text: str, header: str, next_headers: list[str]) -> str:
    """Вытащить markdown-секцию между header и любым из next_headers."""
    if not isinstance(text, str) or header not in text:
        return ""

    start = text.find(header)
    start = text.find("\n", start)
    if start == -1:
        return ""

    body = text[start + 1 :]
    end_positions = [body.find(h) for h in next_headers if h in body]
    end_positions = [pos for pos in end_positions if pos >= 0]
    end = min(end_positions) if end_positions else len(body)
    return body[:end].strip()


def _extract_bullets(section_text: str, limit: int = 3) -> list[str]:
    bullets = []
    for line in (section_text or "").splitlines():
        stripped = line.strip()
        if stripped.startswith("- "):
            bullets.append(stripped[2:].strip())
        elif re.match(r"^\d+\.\s+", stripped):
            bullets.append(re.sub(r"^\d+\.\s+", "", stripped).strip())

        if len(bullets) >= limit:
            break
    return bullets


def _analysis_digest_from_text(name: str, content: str) -> dict:
    footer = _extract_last_json_block(content)

    prompt_summary = _extract_section(
        content,
        "### Prompt summary",
        ["### Strengths", "### Weaknesses", "### Rubric assessment"],
    )

    strengths_section = _extract_section(
        content,
        "### Strengths",
        ["### Weaknesses", "### Rubric assessment"],
    )
    weaknesses_section = _extract_section(
        content,
        "### Weaknesses",
        ["### Rubric assessment", "### Reusable patterns"],
    )
    reusable_section = _extract_section(
        content,
        "### Reusable patterns",
        ["### Improvement recommendations", "### Compact structured block"],
    )

    top_strengths = footer.get("top_strengths") or _extract_bullets(strengths_section, limit=3)
    top_weaknesses = footer.get("top_weaknesses") or _extract_bullets(weaknesses_section, limit=3)
    recommended_actions = footer.get("recommended_actions") or _extract_bullets(
        _extract_section(
            content,
            "### Improvement recommendations",
            ["### Compact structured block"],
        ),
        limit=3,
    )

    payload = {
        "analysis_name": _safe_name(name),
        "file_path": footer.get("file_path"),
        "overall_verdict": footer.get("overall_verdict"),
        "prompt_summary": _clip(prompt_summary, 400),
        "top_strengths": top_strengths[:3],
        "top_weaknesses": top_weaknesses[:3],
        "reusable_patterns": _extract_bullets(reusable_section, limit=3),
        "recommended_actions": recommended_actions[:3],
    }
    return payload


def _report_digest_from_text(name: str, content: str) -> dict:
    footer = _extract_last_json_block(content)

    executive_summary = _extract_section(
        content,
        "## Executive summary",
        ["## Recurrent strengths", "## Recurrent weaknesses", "## Cross-file comparison table"],
    )
    best_practices = _extract_section(
        content,
        "## Best practices extracted",
        ["## Recommendations", "## Report footer", "## Footer JSON"],
    )
    recommendations = _extract_section(
        content,
        "## Recommendations",
        ["## Report footer", "## Footer JSON"],
    )

    payload = {
        "report_name": _safe_name(name),
        "executive_summary": _clip(executive_summary, 500),
        "best_practices": _extract_bullets(best_practices, limit=5),
        "recommendations": _extract_bullets(recommendations, limit=5),
        "footer": footer,
    }
    return payload


# ---- save_analysis ----
def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    safe_name = _safe_name(name)
    path = ANALYSES_DIR / f"{safe_name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Анализ сохранён: {path.name} ({len(content)} символов)"


# ---- list_analyses ----
def list_analyses() -> str:
    """
    Получить список сохранённых анализов.
    Возвращаем компактный JSON-инвентарь, чтобы не расходовать токены зря.
    """
    files = sorted(ANALYSES_DIR.glob("*.md"))
    if not files:
        return json.dumps(
            {
                "count": 0,
                "analyses": [],
                "message": "Нет сохранённых анализов. Сначала вызови save_analysis(name, content).",
            },
            ensure_ascii=False,
            indent=2,
        )

    payload = {
        "count": len(files),
        "analyses": [
            {
                "name": f.stem,
                "bytes": f.stat().st_size,
            }
            for f in files
        ],
        "hint": "Для больших наборов используй read_analysis(names=[...]) батчем.",
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# ---- read_analysis ----
def read_analysis(name: str | None = None, names: list[str] | None = None) -> str:
    """
    Прочитать один или несколько анализов.
    По умолчанию возвращаем компактные digest-версии, а не весь markdown,
    чтобы analysis_report не раздувал контекст.
    """
    target_names = []
    if isinstance(names, list) and names:
        target_names.extend(names)
    if name:
        target_names.append(name)

    target_names = [_safe_name(x) for x in target_names if str(x).strip()]
    if not target_names:
        return json.dumps(
            {"error": "Нужно передать `name` или `names`."},
            ensure_ascii=False,
            indent=2,
        )

    digests = []
    missing = []

    for safe_name in target_names:
        path = ANALYSES_DIR / f"{safe_name}.md"
        if not path.exists():
            missing.append(safe_name)
            continue

        content = path.read_text(encoding="utf-8")
        digests.append(_analysis_digest_from_text(safe_name, content))

    payload = {
        "requested": target_names,
        "returned": len(digests),
        "missing": missing,
        "analyses": digests,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# ---- save_report ----
def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    safe_name = _safe_name(name)
    path = REPORTS_DIR / f"{safe_name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Отчёт сохранён: {path.name} ({len(content)} символов)"


# ---- read_report ----
def read_report(name: str | None = None, names: list[str] | None = None) -> str:
    """
    Прочитать один или несколько отчётов.
    Возвращаем компактный digest, которого достаточно для prompt_generator.
    """
    target_names = []
    if isinstance(names, list) and names:
        target_names.extend(names)
    if name:
        target_names.append(name)

    target_names = [_safe_name(x) for x in target_names if str(x).strip()]
    if not target_names:
        return json.dumps(
            {"error": "Нужно передать `name` или `names`."},
            ensure_ascii=False,
            indent=2,
        )

    reports = []
    missing = []

    for safe_name in target_names:
        path = REPORTS_DIR / f"{safe_name}.md"
        if not path.exists():
            missing.append(safe_name)
            continue

        content = path.read_text(encoding="utf-8")
        reports.append(_report_digest_from_text(safe_name, content))

    payload = {
        "requested": target_names,
        "returned": len(reports),
        "missing": missing,
        "reports": reports,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def _extract_text_from_mcp_result(result) -> str:
    """Достать текст из MCP ToolResult."""
    if result is None:
        return "{}"

    content = getattr(result, "content", None)
    if not content:
        return "{}"

    chunks = []
    for item in content:
        if hasattr(item, "text") and item.text is not None:
            chunks.append(item.text)
        else:
            chunks.append(str(item))
    return "\n".join(chunks) if chunks else "{}"


def _coerce_json(text: str):
    """Попробовать преобразовать текст в Python-объект."""
    if not isinstance(text, str):
        return text

    stripped = text.strip()
    if not stripped:
        return {}

    try:
        return json.loads(stripped)
    except Exception:
        pass

    fenced = re.findall(r"```(?:json)?\s*(.*?)```", stripped, flags=re.DOTALL | re.IGNORECASE)
    for block in fenced:
        try:
            return json.loads(block.strip())
        except Exception:
            continue

    return stripped


def _normalize_entries(data, parent_path: str = "") -> list[dict]:
    """
    Нормализовать ответы directory listing в список объектов с ключами:
    path, name, type, size
    """
    if isinstance(data, dict):
        if isinstance(data.get("entries"), list):
            data = data["entries"]
        elif isinstance(data.get("items"), list):
            data = data["items"]
        elif isinstance(data.get("files"), list):
            data = data["files"]
        elif isinstance(data.get("tree"), list):
            data = data["tree"]
        else:
            return []

    if not isinstance(data, list):
        return []

    entries = []
    for item in data:
        if not isinstance(item, dict):
            continue

        item_path = item.get("path") or item.get("name") or ""
        if parent_path and item_path and "/" not in item_path:
            item_path = f"{parent_path.rstrip('/')}/{item_path}"

        name = item.get("name") or Path(item_path).name
        item_type = item.get("type") or item.get("kind") or ("dir" if item.get("entries") else None)
        size = item.get("size")

        if not item_path:
            continue

        entries.append(
            {
                "path": item_path,
                "name": name,
                "type": item_type,
                "size": size,
            }
        )
    return entries


# ---- List files in github repo ----
async def list_repo_files(session) -> list[dict]:
    """
    Ищем prompt-подобные файлы в repo через search_code.
    Не ограничиваемся только filename:prompt.md.
    """
    owner = "asgeirtj"
    repo = "system_prompts_leaks"

    queries = [
        "prompt",
        "system",
        "instruction",
    ]

    unique: dict[str, dict] = {}

    for query in queries:
        try:
            result = await session.call_tool(
                "search_code",
                {"q": f"repo:{owner}/{repo} {query}"},
            )

            raw_text = _extract_text_from_mcp_result(result)

            try:
                payload = json.loads(raw_text)
            except json.JSONDecodeError:
                print(f"⚠️ search_code вернул не-JSON для query={query}")
                continue

            items = payload.get("items", [])
            for item in items:
                path = item.get("path")
                name = item.get("name")

                if not path:
                    continue

                lower_path = path.lower()

                if not any(ext in lower_path for ext in [".md", ".txt", ".prompt"]):
                    continue

                filename = name or path.split("/")[-1]
                folder = "/".join(path.split("/")[:-1])
                size = item.get("size", 0)

                unique[path] = {
                    "path": path,
                    "name": filename,
                    "folder": folder,
                    "size": size,
                }

        except Exception as e:
            print(f"⚠️ search_code query failed: {query} -> {e}")

    files = sorted(unique.values(), key=lambda x: x["path"])
    print(f"📂 list_repo_files: found {len(files)} unique files")
    return files

📂 Директория анализов: C:\Coding\HSE-Agent-Systems_2026\lectures\lecture_3\hw_3_output\analyses
📂 Директория отчётов:  C:\Coding\HSE-Agent-Systems_2026\lectures\lecture_3\hw_3_output\reports


In [12]:
# Опишите каждый инструмент в формате OpenAI function-calling.

FILE_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "save_analysis",
            "description": "Сохранить анализ одного системного промпта в markdown-файл.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя анализа без расширения .md"},
                    "content": {"type": "string", "description": "Полный текст анализа"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_analyses",
            "description": (
                "Получить компактный список всех сохранённых анализов. "
                "Используй перед analysis_report."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_analysis",
            "description": (
                "Прочитать один или несколько сохранённых анализов в компактном digest-формате. "
                "Для больших наборов prefer names=[...]."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя одного анализа без расширения .md"},
                    "names": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Несколько имён анализов для батч-чтения",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": "Сохранить итоговый отчёт или сгенерированный системный промпт.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя отчёта без расширения .md"},
                    "content": {"type": "string", "description": "Полный текст отчёта или промпта"},
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_report",
            "description": (
                "Прочитать один или несколько сохранённых отчётов в компактном digest-формате. "
                "Для prompt_generator обычно достаточно read_report(name='analysis_report')."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Имя одного отчёта без расширения .md"},
                    "names": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Несколько имён отчётов для батч-чтения",
                    },
                },
                "required": [],
            },
        },
    },
]


# ============================================================
# Маппинг локальных инструментов
# ============================================================

LOCAL_TOOLS = {
    "read_skill": read_skill,
    "save_analysis": save_analysis,
    "list_analyses": list_analyses,
    "read_analysis": read_analysis,
    "save_report": save_report,
    "read_report": read_report,
}

print(f"\n🔧 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  • {name}")


🔧 Локальные инструменты агента (6):
  • read_skill
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report


---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`). 
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [13]:
# ============================================================
# Универсальный агент с tool loop
# ============================================================

# ⚠️ Этап анализа всех промптов может быть долгим, т.к. для каждого файла
# выполняется отдельный агентный вызов с несколькими шагами tool loop.
# Для ускорения вместо синхронных последовательных запросов можно создать
# N потоков (asyncio.gather / ThreadPoolExecutor) и запустить анализ
# нескольких промптов параллельно.
# Как это реализовать — идете на поклон к @Rinaaaa_chan


# ============================================================
# Универсальный агент с tool loop
# ============================================================

# ⚠️ Главная идея против раздувания контекста:
# 1) один запуск run_agent() = одна атомарная задача
# 2) пакетный анализ репозитория делаем СНАРУЖИ циклом по файлам
# 3) в историю не тащим полный content из save_analysis/save_report
# 4) после save_analysis/save_report завершаем run без лишнего шага
import base64

SKILL_TO_LOCAL_TOOLS = {
    "prompt_analysis": ["save_analysis"],
    "analysis_report": ["list_analyses", "read_analysis", "save_report"],
    "prompt_generator": ["read_report", "save_report"],
}

SKILL_TO_MCP_TOOLS = {
    "prompt_analysis": ["get_file_contents"],
    "analysis_report": [],
    "prompt_generator": [],
}

TERMINAL_SAVE_TOOLS = {"save_analysis", "save_report"}


def build_base_system_prompt() -> str:
    return f"""Ты агент-аналитик системных промптов.

Правила работы:
- Один запуск агента решает только ОДНУ атомарную задачу.
- Если задача соответствует skill, сначала вызови read_skill(name).
- После загрузки skill следуй процедуре строго шаг за шагом.
- Не выдумывай содержимое файлов GitHub, анализов и отчётов.
- Для GitHub используй только MCP-инструменты.
- Для локальных результатов используй только файловые инструменты.
- Не пытайся анализировать несколько prompt-файлов в одном запуске.
- Для больших наборов анализов используй list_analyses() и затем read_analysis(names=[...]) батчами.
- read_analysis и read_report возвращают компактные digest-версии. Этого достаточно для synthesis.
- Когда вызван save_analysis или save_report, задача считается завершённой. Не нужен дополнительный шаг.
- Когда данных достаточно, дай краткий финальный ответ на русском языке.

═══════════════════════════════════════════
{skills_metadata}
═══════════════════════════════════════════
"""


def _parse_tool_args(raw_args):
    try:
        return json.loads(raw_args) if isinstance(raw_args, str) else (raw_args or {})
    except Exception:
        return {}


def _sanitize_tool_args_for_history(func_name: str, args: dict) -> dict:
    args = dict(args or {})

    if func_name in TERMINAL_SAVE_TOOLS and "content" in args:
        content = str(args["content"])
        args["content"] = f"<omitted:{len(content)} chars>"
        args["content_preview"] = _clip(content, 200)

    return args


def _build_history_assistant_message(msg, parsed_tool_calls: list[tuple]) -> dict:
    history_message = {
        "role": "assistant",
        "content": msg.content or "",
    }

    if parsed_tool_calls:
        history_message["tool_calls"] = []
        for tool_call, args in parsed_tool_calls:
            history_message["tool_calls"].append(
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": json.dumps(
                            _sanitize_tool_args_for_history(tool_call.function.name, args),
                            ensure_ascii=False,
                        ),
                    },
                }
            )

    return history_message


def _build_terminal_answer(func_name: str, args: dict, result_text: str) -> str:
    saved_name = args.get("name", "unknown")

    if func_name == "save_analysis":
        return f"Готово: анализ сохранён под именем `{saved_name}`. {result_text}"

    if func_name == "save_report":
        return f"Готово: отчёт сохранён под именем `{saved_name}`. {result_text}"

    return result_text


async def run_agent(
    question: str,
    max_steps: int = 10,
    verbose: bool = True,
) -> AgentTrace:
    trace = AgentTrace(question=question)

    base_system_prompt = build_base_system_prompt()
    local_schema_map = {schema["function"]["name"]: schema for schema in FILE_TOOL_SCHEMAS}

    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                tools_response = await session.list_tools()
                all_mcp_schemas = {
                    schema["function"]["name"]: schema
                    for schema in format_mcp_tools(tools_response.tools)
                }
                mcp_tool_names = set(all_mcp_schemas.keys())

                active_tools = [READ_SKILL_TOOL_SCHEMA]
                active_tool_names = {"read_skill"}
                loaded_skills = set()

                messages = [
                    {"role": "system", "content": base_system_prompt},
                    {"role": "user", "content": question},
                ]

                if verbose:
                    print(f"[Agent] Вопрос: {question}")
                    print(f"[Agent] MCP tools available on server: {len(mcp_tool_names)}")
                    print(f"[Agent] Active tools at start: {sorted(active_tool_names)}")

                prev_context_size = 0

                for step_num in range(1, max_steps + 1):
                    current_step = AgentStep(step_number=step_num)

                    if verbose:
                        print("\n" + "=" * 60)
                        print(f"Шаг {step_num}")
                        print(f"[Tools] Active: {sorted(active_tool_names)}")

                    response = client.chat.completions.create(
                        model=MODEL,
                        messages=messages,
                        tools=active_tools,
                        tool_choice="auto",
                        temperature=0,
                    )

                    u = _extract_usage(response)
                    current_step.prompt_tokens = u["prompt_tokens"]
                    current_step.cached_tokens = u["cached_tokens"]
                    current_step.context_size = u["context_size"]
                    current_step.context_size_delta = current_step.context_size - prev_context_size
                    current_step.completion_tokens = u["completion_tokens"]
                    current_step.total_tokens = u["total_tokens"]

                    trace.prompt_tokens_total += current_step.prompt_tokens
                    trace.completion_tokens_total += current_step.completion_tokens
                    trace.total_tokens_total += current_step.total_tokens
                    trace.latest_context_size = current_step.context_size
                    trace.max_context_size = max(trace.max_context_size, current_step.context_size)
                    prev_context_size = current_step.context_size

                    if verbose:
                        print(
                            f"[Tokens] context={current_step.context_size} "
                            f"(Δ {current_step.context_size_delta:+}), "
                            f"completion={current_step.completion_tokens}"
                        )

                    msg = response.choices[0].message

                    if not msg.tool_calls:
                        current_step.final_answer = msg.content or ""
                        trace.steps.append(current_step)
                        trace.total_steps = step_num
                        trace.final_answer = msg.content or ""
                        if verbose:
                            print("[Decision] Финальный ответ")
                            print(f"[Result] {trace.final_answer}")
                        return trace

                    parsed_tool_calls = []
                    for tool_call in msg.tool_calls:
                        args = _parse_tool_args(tool_call.function.arguments or "{}")
                        parsed_tool_calls.append((tool_call, args))

                    messages.append(_build_history_assistant_message(msg, parsed_tool_calls))

                    if verbose:
                        print(f"[Decision] tool_calls: {len(parsed_tool_calls)}")

                    terminal_save_event = None

                    for tool_call, args in parsed_tool_calls:
                        func_name = tool_call.function.name
                        current_step.tool_calls.append({"name": func_name, "args": args})

                        if func_name == "read_skill":
                            skill_name = args.get("name", "")

                            if skill_name in loaded_skills:
                                result_text = f"skill `{skill_name}` already loaded"
                                if verbose:
                                    print(f"[Action] read_skill({skill_name})")
                                    print(f"[Observation] {result_text}")
                            else:
                                result_text = read_skill(skill_name)
                                loaded_skills.add(skill_name)

                                newly_added = []

                                # Локальные инструменты по skill
                                for tool_name in SKILL_TO_LOCAL_TOOLS.get(skill_name, []):
                                    if tool_name in local_schema_map and tool_name not in active_tool_names:
                                        active_tools.append(local_schema_map[tool_name])
                                        active_tool_names.add(tool_name)
                                        newly_added.append(tool_name)

                                # MCP инструменты по skill
                                for tool_name in SKILL_TO_MCP_TOOLS.get(skill_name, []):
                                    if tool_name in all_mcp_schemas and tool_name not in active_tool_names:
                                        active_tools.append(all_mcp_schemas[tool_name])
                                        active_tool_names.add(tool_name)
                                        newly_added.append(tool_name)

                                if verbose:
                                    print(f"[Action] read_skill({skill_name})")
                                    print(f"[Observation] skill loaded, chars={len(result_text)}")
                                    if newly_added:
                                        print(f"[Tools+] disclosed: {newly_added}")

                        elif func_name in LOCAL_TOOLS:
                            try:
                                handler = LOCAL_TOOLS[func_name]
                                result_text = handler(**args) if args else handler()
                            except Exception as e:
                                result_text = json.dumps(
                                    {"error": f"Local tool failed: {func_name}", "details": str(e)},
                                    ensure_ascii=False,
                                )

                            if verbose:
                                print(f"[Action] {func_name}({args})")
                                print(f"[Observation] {_clip(str(result_text), 300)}")

                        elif func_name in mcp_tool_names:
                            try:
                                result = await session.call_tool(func_name, args)

                                if func_name == "get_file_contents":
                                    result_text = extract_github_file_text(result)
                                else:
                                    result_text = _extract_text_from_mcp_result(result)

                            except Exception as e:
                                result_text = json.dumps(
                                    {"error": f"MCP tool failed: {func_name}", "details": str(e)},
                                    ensure_ascii=False,
                                )

                            if verbose:
                                print(f"[Action] MCP::{func_name}({args})")
                                print(f"[Observation] {_clip(result_text, 500)}")

                        else:
                            result_text = json.dumps(
                                {"error": f"Unknown tool: {func_name}", "args": args},
                                ensure_ascii=False,
                            )
                            if verbose:
                                print(f"[Action] {func_name}({args})")
                                print(f"[Observation] {result_text}")

                        parsed_observation = _coerce_json(result_text)
                        current_step.observations.append(parsed_observation)

                        messages.append(
                            {
                                "role": "tool",
                                "tool_call_id": tool_call.id,
                                "content": result_text,
                            }
                        )

                        if func_name in TERMINAL_SAVE_TOOLS:
                            terminal_save_event = (func_name, args, result_text)

                    trace.steps.append(current_step)

                    if verbose:
                        print(f"[State] messages={len(messages)}")

                    if terminal_save_event is not None:
                        func_name, args, result_text = terminal_save_event
                        trace.total_steps = step_num
                        trace.final_answer = _build_terminal_answer(func_name, args, result_text)
                        if verbose:
                            print("[Decision] Терминальное сохранение")
                            print(f"[Result] {trace.final_answer}")
                        return trace

    trace.total_steps = max_steps
    trace.final_answer = "[Агент не завершил работу за отведённое число шагов]"
    return trace


def extract_github_file_text(result) -> str:
    """
    Извлечь именно текст файла из ответа get_file_contents.
    Поддерживает сырой text, JSON с content и base64.
    """
    raw_text = _extract_text_from_mcp_result(result)

    if not raw_text.strip().startswith("{"):
        return raw_text

    try:
        payload = json.loads(raw_text)
    except json.JSONDecodeError:
        return raw_text

    if isinstance(payload, dict):
        content = payload.get("content")
        encoding = payload.get("encoding")

        if isinstance(content, str):
            if encoding == "base64":
                try:
                    return base64.b64decode(content).decode("utf-8", errors="replace")
                except Exception:
                    return content
            return content

        if "text" in payload and isinstance(payload["text"], str):
            return payload["text"]

    if isinstance(payload, dict) and isinstance(payload.get("contents"), list):
        parts = []
        for item in payload["contents"]:
            if not isinstance(item, dict):
                continue
            content = item.get("content")
            encoding = item.get("encoding")
            if isinstance(content, str):
                if encoding == "base64":
                    try:
                        parts.append(base64.b64decode(content).decode("utf-8", errors="replace"))
                    except Exception:
                        parts.append(content)
                else:
                    parts.append(content)
        if parts:
            return "\n\n".join(parts)

    return raw_text

TEST

In [24]:
# ============================================================
# Получить все доступные prompt-like paths для основного прогона
# ============================================================

async def get_all_prompt_paths() -> list[str]:
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                repo_files = await list_repo_files(session)

    paths = [item["path"] for item in repo_files]

    # Лёгкая зачистка явного мусора
    excluded_markers = [
        "/tools.md",
        "tools.md",
        "readme",
        "changelog",
    ]

    filtered = []
    for path in paths:
        p = path.lower()
        if any(marker in p for marker in excluded_markers):
            continue
        filtered.append(path)

    print(f"📂 Всего найдено файлов через discovery: {len(paths)}")
    print(f"📂 После фильтрации: {len(filtered)}")
    return filtered

In [25]:
def clear_hw3_outputs():
    for path in ANALYSES_DIR.glob("*.md"):
        path.unlink()
    for path in REPORTS_DIR.glob("*.md"):
        path.unlink()
    print("✅ hw_3_output очищен")


In [26]:
# ============================================================
# Тест кейс 1. Запрос на начало анализа
# ============================================================

clear_hw3_outputs()

TEST_CASE_1_PATHS = await get_all_prompt_paths()

print("📜 TEST CASE 1: prompt_analysis")
print(f"Будем анализировать {len(TEST_CASE_1_PATHS)} файл(ов):")
for p in TEST_CASE_1_PATHS[:20]:
    print(" •", p)

if len(TEST_CASE_1_PATHS) > 20:
    print(f"... и ещё {len(TEST_CASE_1_PATHS) - 20} файлов")

tc1_runs = await run_prompt_analysis_batch(TEST_CASE_1_PATHS)

saved_analyses = sorted(p.stem for p in ANALYSES_DIR.glob("*.md"))
print("\n✅ Сохранённые анализы:")
print(f"Всего: {len(saved_analyses)}")

assert len(saved_analyses) == len(TEST_CASE_1_PATHS), (
    f"Ожидали {len(TEST_CASE_1_PATHS)} анализов, получили {len(saved_analyses)}"
)

# Не печатаем summary для каждого файла - иначе ноутбук раздуется
print("\n=== TC1 RESULT ===")
print(f"Processed files: {len(TEST_CASE_1_PATHS)}")
print(f"Saved analyses: {len(saved_analyses)}")

✅ hw_3_output очищен
📂 list_repo_files: found 71 unique files
📂 Всего найдено файлов через discovery: 71
📂 После фильтрации: 66
📜 TEST CASE 1: prompt_analysis
Будем анализировать 66 файл(ов):
 • Anthropic/FlintK12/prompt.md
 • Anthropic/claude-opus-4.6.md
 • Anthropic/claude-sonnet-4.6.md
 • Anthropic/claude.ai-human-readable.md
 • Anthropic/old/claude-3.7-sonnet-full-system-message-humanreadable.md
 • Anthropic/old/claude-4.1-opus-thinking.md
 • Anthropic/old/claude-4.5-sonnet.md
 • Anthropic/old/claude-opus-4.5.md
 • Anthropic/old/claude-sonnet-4.md
 • Anthropic/raw/claude-opus-4.6-no-tools-raw.md
 • Anthropic/raw/claude-opus-4.6-raw.md
 • Anthropic/raw/claude-sonnet-4.6-no-tools-raw.md
 • Anthropic/raw/claude-sonnet-4.6-raw.md
 • Anthropic/visualize.md
 • Google/Gemini CLI System.md
 • Google/NotebookLM-chat.md
 • Google/ai-studio-build.md
 • Google/gemini-2.5-flash-image-preview.md
 • Google/gemini-2.5-pro-api.md
 • Google/gemini-2.5-pro-guided-learning.md
... и ещё 46 файлов
📋 Буд

In [27]:
# ============================================================
# Тест кейс 2. Запрос на формирование итоговой сводки
# ============================================================

print("📜 TEST CASE 2: analysis_report")

tc2_trace = await run_agent(
    """Нужно сформировать итоговую сводку по всем уже сохранённым анализам.

Используй skill `analysis_report`.

Что нужно сделать:
1. Загрузи skill `analysis_report`
2. Получи список анализов через list_analyses()
3. Прочитай все доступные анализы через read_analysis(name)
4. Сформируй сводный отчёт с повторяющимися паттернами, различиями и рекомендациями
5. Сохрани отчёт через save_report(name, content) с именем `analysis_report`
6. Верни краткий итог
""",
    max_steps=8,
    verbose=True,
)

report_path = REPORTS_DIR / "analysis_report.md"
assert report_path.exists(), f"Не найден файл отчёта: {report_path}"

print("\n=== TC2 SUMMARY ===")
print(tc2_trace.summary())

print("\n=== REPORT PREVIEW ===")
print(report_path.read_text(encoding="utf-8")[:4000])

📜 TEST CASE 2: analysis_report
[Agent] Вопрос: Нужно сформировать итоговую сводку по всем уже сохранённым анализам.

Используй skill `analysis_report`.

Что нужно сделать:
1. Загрузи skill `analysis_report`
2. Получи список анализов через list_analyses()
3. Прочитай все доступные анализы через read_analysis(name)
4. Сформируй сводный отчёт с повторяющимися паттернами, различиями и рекомендациями
5. Сохрани отчёт через save_report(name, content) с именем `analysis_report`
6. Верни краткий итог

[Agent] MCP tools available on server: 26
[Agent] Active tools at start: ['read_skill']

Шаг 1
[Tools] Active: ['read_skill']
[Tokens] context=706 (Δ +706), completion=15
[Decision] tool_calls: 1
[Action] read_skill(analysis_report)
[Observation] skill loaded, chars=2357
[Tools+] disclosed: ['list_analyses', 'read_analysis', 'save_report']
[State] messages=4

Шаг 2
[Tools] Active: ['list_analyses', 'read_analysis', 'read_skill', 'save_report']
[Tokens] context=1395 (Δ +689), completion=12
[Decisi

In [28]:
# ============================================================
# Тест кейс 3. Запрос на формирование промпта для случайного ассистента
# ============================================================

print("📜 TEST CASE 3: prompt_generator")

tc3_trace = await run_agent(
    """Нужно сгенерировать новый системный промпт для customer support assistant.

Используй skill `prompt_generator`.

Входные данные:
- report_name: analysis_report
- output_name: generated_prompt

Что нужно сделать:
1. Загрузи skill `prompt_generator`
2. Прочитай отчёт через read_report(name="analysis_report")
3. На основе отчёта сгенерируй новый системный промпт
4. Сохрани результат через save_report(name, content) с именем `generated_prompt`
5. Верни краткий итог
""",
    max_steps=8,
    verbose=True,
)

generated_prompt_path = REPORTS_DIR / "generated_prompt.md"
assert generated_prompt_path.exists(), f"Не найден сгенерированный промпт: {generated_prompt_path}"

print("\n=== TC3 SUMMARY ===")
print(tc3_trace.summary())

print("\n=== GENERATED PROMPT PREVIEW ===")
print(generated_prompt_path.read_text(encoding="utf-8")[:4000])

📜 TEST CASE 3: prompt_generator
[Agent] Вопрос: Нужно сгенерировать новый системный промпт для customer support assistant.

Используй skill `prompt_generator`.

Входные данные:
- report_name: analysis_report
- output_name: generated_prompt

Что нужно сделать:
1. Загрузи skill `prompt_generator`
2. Прочитай отчёт через read_report(name="analysis_report")
3. На основе отчёта сгенерируй новый системный промпт
4. Сохрани результат через save_report(name, content) с именем `generated_prompt`
5. Верни краткий итог

[Agent] MCP tools available on server: 26
[Agent] Active tools at start: ['read_skill']

Шаг 1
[Tools] Active: ['read_skill']
[Tokens] context=703 (Δ +703), completion=15
[Decision] tool_calls: 1
[Action] read_skill(prompt_generator)
[Observation] skill loaded, chars=2173
[Tools+] disclosed: ['read_report', 'save_report']
[State] messages=4

Шаг 2
[Tools] Active: ['read_report', 'read_skill', 'save_report']
[Tokens] context=1328 (Δ +625), completion=15
[Decision] tool_calls: 1
[Ac

<img src="pictures\hw_file_1.PNG" width="600"/>

<img src="pictures\hw_file_2.PNG" width="600"/>